# Synthetic Data Generation Tutorial using LLaMA and Mixtral

This tutorial demonstrates how to use SDG repository to generate synthetic question-answer pairs from documents using LLaMA 7b model. 

1. Setting up the environment
2. Connecting to LLM servers
3. Configuring the data generation pipeline
4. Generating data with different models
5. Comparing results

In [ ]:
# Enable auto-reloading of modules - useful during development
%load_ext autoreload
%autoreload 2

### Setup Instructions

#Before running this notebook, you'll need to install latest version of stable SDG or from the repo

```bash 
pip install git+https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
pip install datasets
pip install openai
```

In [2]:
!ls

data-generation-with-llama-70b.ipynb  synth_knowledge1.5_llama3.3.yaml
data-generation-with-llama-7b.ipynb   synth_knowledge1.5_llama-7b.yaml


In [6]:
 import sys
 sys.path.insert(0, '/home/akamra/sdg_hub/src')

In [7]:
# Import required libraries
# datasets: For handling our data
# OpenAI: For interfacing with the LLM servers
# SDG components: For building our data generation pipeline
from datasets import load_dataset, Dataset
from openai import OpenAI

from sdg_hub.flow import Flow
from sdg_hub.pipeline import Pipeline
from sdg_hub.sdg import SDG
from sdg_hub.registry import PromptRegistry

In [8]:
# Configure OpenAI client to connect to our local vLLM server
endpoint = f"http://localhost:8001/v1"
openai_api_key = "EMPTY"  # vLLM doesn't require real API key
openai_api_base = endpoint

client = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base,
)

# Verify we can see the model
teacher_model = client.models.list().data[0].id
print(f"Connected to model: {teacher_model}")

[17:18:13] INFO     HTTP Request: GET http://localhost:8001/v1/models "HTTP/1.1 200 OK"             ]8;id=136744;file:///home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=281219;file:///home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/httpx/_client.py#1026\1026]8;;\

Connected to model: /model/RedHatAI/Meta-Llama-3.1-8B-Instruct-quantized.w4a16


### Configure the Data Generation Pipeline

Now we'll set up our Synthetic Data Generation (SDG) pipeline with the following components:
1. SDG Flow configuration from YAML
2. SDG Pipeline setup
3. SDG configuration with batch processing, number of workers, and save frequency parameters

In [12]:
# Load the flow configuration from YAML file
flow_cfg = Flow(client).get_flow_from_file("synth_knowledge1.5_llama-7b.yaml")

# Initialize the SDG pipeline with processing parameters
sdg = SDG(
    [flow_cfg],
    num_workers=1,      # Number of parallel workers
    batch_size=1,       # Batch size for processing
    save_freq=1000,     # How often to save checkpoints
)

### Load and Prepare Seed Data

We'll load our seed data (documents) that will be used to generate question-answer pairs.

In [13]:
# Load the seed data from JSON file
seed_data_path = "/home/akamra/sdg_hub/examples/knowledge_tuning/instructlab/document_collection/ibm-annual-report/ibm-annual-report-2024.json"  # Replace with your data path
ds = load_dataset('json', data_files=seed_data_path, split='train')

# For testing, we'll use just one example
ds = ds.select(range(1))

### Generate Data with LLaMA 3.3

Now we'll use our configured pipeline to generate synthetic question-answer pairs.

In [14]:
# Generate synthetic data and save checkpoints
generated_data = sdg.generate(ds, checkpoint_dir="Tmp")

[17:25:47] INFO     No existing checkpoints found in Tmp, generating from scratch                ]8;id=952180;file:///home/akamra/sdg_hub/src/sdg_hub/checkpointer.py\checkpointer.py]8;;\:]8;id=775157;file:///home/akamra/sdg_hub/src/sdg_hub/checkpointer.py#72\72]8;;\

           INFO     Splitting the dataset into smaller batches                                           ]8;id=454903;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=590278;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#167\167]8;;\

100%|██████████| 1/1 [00:00<00:00, 28339.89it/s]


           INFO     Generating dataset with 1 splits, batch size 1, and 1 workers                        ]8;id=159148;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=203211;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#169\169]8;;\

           INFO     Processing split 0                                                                   ]8;id=629070;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=668479;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#116\116]8;;\

  0%|          | 0/1 [00:00<?, ?it/s]

           INFO     🔄 Running block 1/13: duplicate_document_col                                       ]8;id=767041;file:///home/akamra/sdg_hub/src/sdg_hub/flow.py\flow.py]8;;\:]8;id=985006;file:///home/akamra/sdg_hub/src/sdg_hub/flow.py#212\212]8;;\

           ERROR    Error processing split 0: "Column document not in the dataset. Current columns in    ]8;id=448415;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py\sdg.py]8;;\:]8;id=542685;file:///home/akamra/sdg_hub/src/sdg_hub/sdg.py#123\123]8;;\
                    the dataset: ['_name', 'type', 'description', 'file-info', 'main-text', 'figures',             
                    'tables', 'equations', 'footnotes', 'page-dimensions', 'page-footers',                         
                    'page-headers']"                                                                               

Traceback (most recent call last):
  File "/home/akamra/sdg_hub/src/sdg_hub/sdg.py", line 120, in _generate_data
    input_split = flow.generate(input_split)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/akamra/sdg_hub/src/sdg_hub/flow.py", line 222, in generate
    dataset = block.generate(dataset, **gen_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/akamra/sdg_hub/src/sdg_hub/blocks/utilblocks.py", line 449, in generate
    self.columns_map[col_to_dup], samples[col_to_dup]
                                  ~~~~~~~^^^^^^^^^^^^
  File "/home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/datasets/arrow_dataset.py", line 2777, in __getitem__
    return self._getitem(key)
           ^^^^^^^^^^^^^^^^^^
  File "/home/akamra/.var/app/org.jupyter.JupyterLab/config/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/datasets/arrow_dataset.py", line 2761, in _getitem
    pa_subtable = que

### Setting up Mixtral Model

For comparison, we'll also generate data using the Mixtral model. First, start the Mixtral server:

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3,4,5,6,7 python -m vllm.entrypoints.openai.api_server \
    --model meta-llama/Llama-3.3-70B-Instruct \
    --dtype float16 \
    --tensor-parallel-size 8 
```

In [ ]:
# Connect to Mixtral model running on a different server
mistral_client = OpenAI(
    api_key="EMPTY",
    base_url=f"http://10.7.0.15:8000/v1",  # Update with your Mixtral server address
)

# Verify connection to Mixtral model
mistral_client_teacher_model = mistral_client.models.list().data[0].id
print(f"Connected to Mixtral model: {mistral_client_teacher_model}")

### Configure Mixtral Pipeline

Set up a similar pipeline for Mixtral model generation.

In [ ]:
# Create flow configuration for Mixtral
flow_cfg_mistral = Flow(mistral_client).get_flow_from_file(
    "../../src/sdg_hub/flows/generation/knowledge/synth_knowledge1.5.yaml"
)

# Initialize SDG pipeline for Mixtral
sdg_mistral = SDG(
    [Pipeline(flow_cfg_mistral)],
    num_workers=1,
    batch_size=1,
    save_freq=1000,
)

### Generate Data with Mixtral

Generate synthetic data using the Mixtral model for comparison.

In [ ]:
# Generate data using Mixtral model
generated_data_mistral = sdg_mistral.generate(ds, checkpoint_dir="Tmp")

### Compare Generated Data

Let's compare the outputs from both models by saving them to a markdown file for easy review.

In [ ]:
# Save comparison results to markdown file
k = 5  # Number of examples to compare
output_file = "model_comparison.md"

with open(output_file, "w") as f:
    # Write the source document first
    f.write(f"### Document \n{generated_data[0]['document']}")
    
    # Compare generated Q&A pairs
    for i in range(min(len(generated_data), len(generated_data_mistral))):
        f.write("Example #{}\n".format(i+1))
        
        # LLaMA 3.3 results
        f.write("### Result from llama3.3\n")
        f.write(generated_data[i]['question'] + "\n")
        f.write("*******************************\n")
        f.write(generated_data[i]['response'] + "\n")
        f.write("=================================\n")
        
        # Mixtral results
        f.write("### Result from mistral\n") 
        f.write(generated_data_mistral[i]['question'] + "\n")
        f.write("*******************************\n")
        f.write(generated_data_mistral[i]['response'] + "\n")
        f.write("\n\n")

print(f"Wrote {k} examples to {output_file}")

### Production Usage

For large-scale data generation, use the command-line script instead of this notebook:

```bash
python scripts/generate.py --ds_path seed_data.jsonl \
    --bs 2 --num_workers 10 \
    --save_path <your_save_path> \
    --flow ../src/sdg_hub/flows/generation/knowledge/synth_knowledge1.5.yaml \
    --checkpoint_dir <your_checkpoint_dir> \
    --endpoint <your_endpoint>
```

Note: For LLaMA 3.3, use `synth_knowledge1.5_llama3.3.yaml` as the flow configuration file.